# UpSet (interactive)

> The same UpSet layout as [UpSet (static)](upset.html), rendered in Plotly with members shown on hover.

In [1]:
#| default_exp upset_interactive

In [2]:
#| export
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from upsetplot import UpSet, from_contents
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
#| export
def _pack(items, max_rows, gutter=3):
    cols = max(1, -(-len(items) // max_rows))
    rows = -(-len(items) // cols)
    col_items = [items[c * rows:(c + 1) * rows] for c in range(cols)]
    widths = [max((len(i) for i in col), default=0) + gutter for col in col_items]
    lines = []
    for r in range(rows):
        line = "".join(col_items[c][r].ljust(widths[c]) if r < len(col_items[c]) else " " * widths[c] for c in range(cols))
        lines.append(line.rstrip().replace(" ", chr(160)))
    return "<br>".join(lines)

def upset_interactive(data_dict, color_list=None, bar_width=0.8, totals_bar_height=0.6,
                      matrix_height=1.0, intersection_plot_elements=6, totals_plot_elements=2,
                      circle_size=100, gridkey_fontsize=13, axis_fontsize=13, hover_size=12,
                      count_fontsize=12, count_color=None,
                      figsize=None, font="Inter", sort_by="cardinality",
                      shading=True, shading_color="#f0f0f0",
                      wspace=0, hspace=0, title="", title_fontsize=20, title_fontweight="bold", save_path=None):
    # plotly UpSet: hover any intersection bar to list its members; the tooltip stays below the title.
    # member text lives in the hover tooltip, so only the naming (gridkey_fontsize), the number (count_fontsize)
    # and the tooltip (hover_size) are sized here; there is no in-bar text to auto-fit.
    fam = font + ", sans-serif"
    data_dict = {k: (v.iloc[:, 0].tolist() if isinstance(v, pd.DataFrame) else list(v)) for k, v in data_dict.items()}
    matrix = from_contents(data_dict)
    obj = UpSet(matrix, show_counts=True, sort_by=sort_by)
    sizes = obj.intersections
    cats = list(sizes.index.names)
    R, n = len(cats), len(sizes)
    counts = [int(c) for c in sizes.values]
    signatures = [tuple(bool(b) for b in idx) for idx in sizes.index]
    totals = [int(obj.totals[c]) for c in cats]
    gene_lists = []
    for subset_key in sizes.index:
        mask = True
        for cat_name, is_present in zip(sizes.index.names, subset_key):
            mask &= (matrix.index.get_level_values(cat_name) == is_present)
        genes = matrix[mask]
        gene_lists.append([str(g[0]) if isinstance(g, (list, np.ndarray)) else str(g) for g in genes.values])
    if isinstance(color_list, str):
        cmap = plt.get_cmap(color_list)
        cbase = cmap.colors if hasattr(cmap, "colors") else [cmap(k / max(n - 1, 1)) for k in range(n)]
        color_list = [mpl.colors.to_hex(cbase[k % len(cbase)]) for k in range(n)]
    if color_list is None:
        color_list = ["#333333"] * n

    width, height = (figsize[0] * 72, figsize[1] * 72) if figsize is not None else (max(750, 260 + n * 30), 680)
    dot_size = max(6, circle_size ** 0.5 * 1.4)
    ccol = count_color if count_color is not None else [mpl.colors.to_hex([v * 0.6 for v in mpl.colors.to_rgb(c)]) for c in color_list]
    w0 = min(0.35, max(0.1, 0.08 + totals_plot_elements * 0.03))
    matrix_frac = min(0.6, max(0.1, (R * matrix_height) / (intersection_plot_elements + R * matrix_height)))
    top_h = 1 - matrix_frac
    auto_hs = (max(len(c) for c in cats) * gridkey_fontsize * 0.62 + 15) / width
    hspacing = min(0.4, max(0.03, auto_hs + wspace * 0.08))
    vspacing = min(0.3, max(0.02, 0.05 + hspace * 0.05))
    max_rows = max(3, int((height - 130) * top_h * 0.6 / (hover_size * 1.5)))
    hovers = [_pack(gl, max_rows) if gl else "" for gl in gene_lists]

    fig = make_subplots(rows=2, cols=2, column_widths=[w0, 1 - w0], row_heights=[top_h, matrix_frac],
                        shared_xaxes=True, shared_yaxes=False, horizontal_spacing=hspacing, vertical_spacing=vspacing,
                        specs=[[None, {}], [{}, {}]])
    fig.add_trace(go.Bar(x=list(range(n)), y=counts, width=bar_width, marker_color=color_list,
                         text=[str(c) for c in counts], textposition="outside",
                         textfont=dict(size=count_fontsize, color=ccol, family=fam),
                         customdata=hovers, hovertemplate="%{customdata}<extra></extra>",
                         hoverlabel=dict(bgcolor="rgba(211,211,211,0.5)", font=dict(size=hover_size, color="black", family=fam)),
                         showlegend=False), row=1, col=2)
    bx = [j for j in range(n) for i in range(R)]
    by = [i for j in range(n) for i in range(R)]
    fig.add_trace(go.Scatter(x=bx, y=by, mode="markers", marker=dict(size=dot_size, color="#c4c4c4"), hoverinfo="skip", showlegend=False), row=2, col=2)
    fx, fy, fc, lx, ly = [], [], [], [], []
    for j, sig in enumerate(signatures):
        on = [i for i, o in enumerate(sig) if o]
        for i in on:
            fx.append(j); fy.append(i); fc.append(color_list[j])
        if on:
            lx += [j, j, None]; ly += [min(on), max(on), None]
    fig.add_trace(go.Scatter(x=lx, y=ly, mode="lines", line=dict(color="black", width=dot_size * 0.12), hoverinfo="skip", showlegend=False), row=2, col=2)
    fig.add_trace(go.Scatter(x=fx, y=fy, mode="markers",
                             marker=dict(size=dot_size, color=fc, line=dict(color="black", width=dot_size * 0.12)),
                             hoverinfo="skip", showlegend=False), row=2, col=2)
    fig.add_trace(go.Bar(y=list(range(R)), x=totals, orientation="h", width=totals_bar_height, marker_color="black", hoverinfo="skip", showlegend=False), row=2, col=1)

    if shading:
        myax = fig.data[1].yaxis
        for i in range(0, R, 2):
            fig.add_shape(type="rect", xref="paper", x0=0, x1=1, yref=myax, y0=i - 0.5, y1=i + 0.5,
                          fillcolor=shading_color, line_width=0, layer="below")

    yr = [R - 0.4, -0.6]
    fig.update_yaxes(title_text="Intersection size", range=[0, max(counts) * 1.3], row=1, col=2, showgrid=False,
                     zeroline=False, showline=True, linecolor="black", linewidth=1, ticks="outside", tickcolor="black",
                     tickfont=dict(size=axis_fontsize, color="black"), title_font=dict(size=axis_fontsize, color="black"))
    fig.update_xaxes(range=[-0.6, n - 0.4], showticklabels=False, showgrid=False, row=1, col=2,
                     showline=True, linecolor="black", linewidth=1)
    fig.update_xaxes(range=[-0.6, n - 0.4], showticklabels=False, showgrid=False, zeroline=False, row=2, col=2)
    fig.update_yaxes(tickvals=list(range(R)), ticktext=cats, range=yr, side="left", row=2, col=2, showgrid=False, zeroline=False, tickfont=dict(size=gridkey_fontsize, color="black"))
    fig.update_xaxes(title_text="Set size", autorange="reversed", showgrid=False, row=2, col=1,
                     showline=True, linecolor="black", linewidth=1, ticks="outside", tickcolor="black")
    fig.update_yaxes(range=yr, showticklabels=False, showgrid=False, zeroline=False, row=2, col=1)
    fig.update_layout(title=dict(text=title, font=dict(size=title_fontsize, weight=title_fontweight, color="black", family=fam)),
                      width=width, height=height, plot_bgcolor="white",
                      font=dict(color="black", family=fam), margin=dict(t=90, l=110, r=40, b=60))
    if save_path:
        fig.write_html(save_path)
    fig.show()

In [4]:
#| hide
from matplotlib import font_manager
import matplotlib.pyplot as plt
for f in font_manager.findSystemFonts(fontpaths=["Font" if __import__("os").path.isdir("Font") else "../Font"]):
    font_manager.fontManager.addfont(f)
plt.rcParams["font.family"] = "Inter"

In [5]:
#| hide
import plotly.io as pio
pio.renderers.default = "notebook_connected"

## Data

In [6]:
drugbase = ("Drug target sample files/" if __import__("os").path.isdir("Drug target sample files") else "../Drug target sample files/")
flu = pd.read_csv(drugbase + "1. FLUOXETINE_sample.csv")["Gene names"].dropna().tolist()
ibu = pd.read_csv(drugbase + "2. IBUPROFEN_sample.csv")["Gene names"].dropna().tolist()
acet = pd.read_csv(drugbase + "3. ACETAMINOPHEN_sample.csv")["Gene names"].dropna().tolist()

## Interactive UpSet

Hover any intersection bar to list its members.

In [7]:
upset_interactive({"Fluoxetine": flu, "Ibuprofen": ibu, "Acetaminophen": acet}, title="Drug target intersections", color_list="Set2", figsize=(10.5, 6))

## Controlling plot aesthetics

| Argument | Controls |
|---|---|
| `color_list` | list of bar colours, or a colormap name; one per intersection bar |
| `title`, `title_fontsize` | figure title text and size |
| `count_color`, `count_fontsize` | colour and size of the count above each bar |
| `hover_size` | font size of the member list in the hover tooltip |
| `circle_size` | size of the dot-matrix markers |
| `gridkey_fontsize`, `axis_fontsize` | font sizes of the set names and the axes |
| `bar_width`, `totals_bar_height` | thickness of the intersection and set-total bars |
| `shading`, `shading_color` | the alternating row bands behind the dot matrix |
| `sort_by` | order bars by `"cardinality"` or `"degree"` |
| `figsize` | figure size in inches (converted to pixels) |
| `save_path` | optional path to save a standalone HTML file |

**Colormap names.** Any Matplotlib colormap name works for `color_list`; qualitative maps keep the sets/bars most distinct: `Set1`, `Set2`, `Set3`, `Dark2`, `Paired`, `Accent`, `Pastel1`, `Pastel2`, `tab10`, `tab20`. You can also pass an explicit list of colours instead.